# Matter — commission a bridge, read and switch it

A smart-home bridge (`MatterDevice` + `MatterBridgeSimulator`) and a controller (`MatterController`) share an in-memory network. With real devices use `UseUdp()` and the device's address on UDP 5540; the controller stays read-only until `AllowWrites()`.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.30.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.30.0-preview.1"
#r "nuget: IoTCom.Net.Protocols.Matter, 0.30.0-preview.1"

## Onboarding codes and TLV
The QR payload and the manual code carry the discriminator and passcode; Matter messages carry TLV.

In [ ]:
using System.Net;
using IoTCom.Net;
using IoTCom.Net.Protocols.Matter;
using IoTCom.Net.Transports;

var payload = MatterSetupPayload.FromQrCode("MT:Y.K9042C00KA0648G00");
Console.WriteLine($"{payload.Describe()} -> manual code {payload.ToManualCode()}");
var tlv = Tlv.Encode(TlvElement.Structure(TlvTag.Anonymous, TlvElement.UInt(TlvTag.Context(0), 42), TlvElement.String(TlvTag.Context(1), "Lampu")));
Console.WriteLine($"{Convert.ToHexString(tlv)} = {Tlv.Decode(tlv)}");

## Commission
PASE with the passcode, fail-safe, attestation, CSR, root and NOC, then CASE — each step reported.

In [ ]:
var net = new InMemoryDatagramNetwork();
var at = new IPEndPoint(IPAddress.Parse("10.0.0.70"), 5540);
var bridge = MatterDevice.Create(o => { o.UseInMemory(net, at); o.ProductName = "IoTCom smart home bridge"; });
var house = new MatterBridgeSimulator(bridge);
await bridge.StartAsync();
var controller = MatterController.Create(o => o.UseInMemory(net).AllowWrites().TrustTestDevices());
var node = await controller.CommissionAsync(at, bridge.SetupPayload, 0x42, new Progress<MatterCommissioningStep>(s => Console.WriteLine($"  {s}")));
Console.WriteLine($"node 0x{node.NodeId:X} on fabric 0x{controller.FabricId:X} ({controller.CompressedFabricId})");

## Read, invoke, subscribe
Wildcard reads over the bridged endpoints, a subscription, and an On command.

In [ ]:
foreach (var r in await node.ReadAsync([new AttributePath(null, MatterIds.BridgedDeviceBasicInformation, 5)]))
    Console.WriteLine($"endpoint {r.Path.Endpoint}: {r.Value!.AsString()}");
await node.SubscribeAsync([new AttributePath(null, MatterIds.OnOff, 0)], 0, 60, r => Console.WriteLine($"report {r.Path} = {r.Value}"));
await node.InvokeAsync(4, MatterIds.OnOff, 1);
await Task.Delay(300);
Console.WriteLine($"terrace light on: {house.Lights[2].OnOff.On}");
var readOnly = MatterController.Create(o => o.UseInMemory(net));
try { await readOnly.CommissionAsync(at, bridge.SetupPayload, 9); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
await readOnly.DisposeAsync();
await controller.DisposeAsync();
await bridge.DisposeAsync();

## Going further
`iotcom matter code`, `iotcom matter device --bridge`, `iotcom matter commission <host> <code> --allow-write`, `iotcom matter demo --allow-write`, the Gallery's *Smart home over Matter* and the MatterHome sample. See `docs/en/protocols/matter.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*